# Training Loop

In [ ]:
# Cell 1: Setup autoreload
%load_ext autoreload
%autoreload 2

In [ ]:
# Setup Experiment and run training
from storm_regression.training_pipeline import TrainingConfig, run_training_pipeline
from storm_utils.config_paths import get_project_paths
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
import copy

paths = get_project_paths()

import logging

# Set root logger
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

# Get logger for your notebook/script
logger = logging.getLogger(__name__)

In [ ]:
paths = get_project_paths()
huxt_run_id = 3

config = TrainingConfig(
    huxt_run_id=huxt_run_id,
    huxt_data_path=paths['huxt_data_shared'] / f'HUXt{huxt_run_id}_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / f'HUXt{huxt_run_id}_modified' / 'discontinuities.npy',
    
    model_type = 'ensemble'
    n_ensembles=100,

    ensemble_regressors={
        'LinearRegression': LinearRegression,
        # 'Ridge': lambda: Ridge(alpha=1.0, positive=True),
        # 'DecisionTree': lambda: DecisionTreeRegressor(max_depth=10, random_state=42),
        # 'RandomForest': lambda: RandomForestRegressor(n_estimators=30, max_depth=10),
        # 'XGBoost_Fast': lambda: XGBRegressor(
        #     n_estimators=100,      # Fewer trees (faster)
        #     max_depth=5,           # Shallow trees (faster, less overfit)
        #     learning_rate=0.05,    # Higher LR with fewer trees
        #     subsample=0.8,
        #     colsample_bytree=0.8,
        #     tree_method='hist',    # Histogram-based (much faster!)
        #     random_state=42,
        #     n_jobs=1
        # )
    },
    
    # Update constraints for each model
    model_constraints={
        'LinearRegression': 'realistic',  # Needs constraints
        'Ridge': None,                    # Has positive=True
        'DecisionTree': None,
        'RandomForest': 'realistic',      # May predict negatives
        'XGBoost_Fast': None,                  
    },

    lead_times=[24],
    storm_thresholds=[4.5,],
    random_seeds=[42],
    test_folds=[0],
    omni_subsets=[["Bz_GSM"]],
    balance=False,
    save_results=True,
    n_jobs=-1,  # Use all CPUs
)

# Run training
run_training_pipeline(config)

## MLP

### Phase 1 - Features

In [ ]:
# Setup Experiment and run training for MLP
from storm_regression.training_pipeline import TrainingConfig, run_training_pipeline
from storm_utils.config_paths import get_project_paths
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

paths = get_project_paths()
huxt_run_id = 3

# Experiment 001: Baseline - V only
# Phase 1: Feature exploration
config_phase1 = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    
    model_type='mlp',
    mlp_use_ensemble_statistics=True,
    mlp_n_epochs=100,
    mlp_learning_rate=0.001,
    mlp_patience=10,
    
    lead_times=[12],
    random_seeds=[42],
    test_folds=[0],
    storm_thresholds=[4.5],
    balance=True,
    save_results=True,
    
    # Test all 10 feature combinations in one run
    omni_subsets=[
        [],                           # 1. Baseline: V only
        ['Bz_GSM'],                   # 2. IMF Bz (most important for storms)
        ['Dst'],                      # 3. Ring current index
        ['n_sw'],                     # 4. Density
        ['flow_pressure'],            # 5. Dynamic pressure
        ['Bz_GSM', 'Dst'],           # 6. Best two combined
        ['Bz_GSM', 'n_sw'],          # 7. IMF + density
        ['Bz_GSM', 'flow_pressure'], # 8. IMF + pressure
        ['Bz_GSM', 'Dst', 'n_sw'],   # 9. Top three
        ['Bz_GSM', 'Dst', 'flow_pressure', 'E_field'],  # 10. Kitchen sink
    ],
    
    experiment_phase='phase1_features',
)

run_training_pipeline(config_phase1)

### Phase 1 - Ensemble Spread

In [ ]:
# Setup Experiment and run training for MLP
from storm_regression.training_pipeline import TrainingConfig, run_training_pipeline
from storm_utils.config_paths import get_project_paths
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

paths = get_project_paths()
huxt_run_id = 3

# Phase 1.2: Ensemble spread exploration
config_phase1_ensemble_spread = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    
    model_type='mlp',
    mlp_include_ensemble_spread=False,
    # mlp_ensemble_percentiles=[50], # don't alter this param within experiment loop
    mlp_n_epochs=200,
    mlp_learning_rate=0.005,
    mlp_patience=10,
    
    lead_times=[12],
    random_seeds=[42],
    test_folds=[0],
    storm_thresholds=[4.5],
    balance=True,
    save_results=True,
    
    omni_subset=[], # Don't use any omni to purely visualise the 

    experiment_phase='phase1_ensemble_spread',
)

In [ ]:
mlp_ensemble_percentiles = [
    # Basic configurations
    [50],                           # 1. Median only (minimal)
    [5, 50, 95],                    # 2. Baseline: median + 90% bounds
    [25, 50, 75],                   # 3. Interquartile range
    
    # More detailed distributions
    [5, 25, 50, 75, 95],            # 4. Quintiles (5 points)
    [10, 30, 50, 70, 90],           # 5. Alternative quintiles
    [5, 16, 50, 84, 95],            # 6. Roughly ±1σ, ±2σ for normal
    
    # Very detailed
    [5, 16, 33, 50, 67, 84, 95],    # 7. Seven percentiles
    [1, 10, 25, 50, 75, 90, 99],    # 8. Extreme tails included
    
    # Asymmetric (more detail in tails)
    [1, 5, 25, 50, 75, 95, 99],     # 9. Focus on extremes
    [5, 10, 25, 50, 95],            # 10. Asymmetric (sparse upper tail)
    ]

for ens_p in mlp_ensemble_percentiles: 
    config_phase1_ensemble_spread.mlp_ensemble_percentiles = ens_p
    run_training_pipeline(config_phase1_ensemble_spread)

### Phase 2 - Architecture taper

In [ ]:
import logging

# Set root logger
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

# Get logger for your notebook/script
logger = logging.getLogger(__name__)

# Setup Experiment and run training for MLP
from storm_regression.training_pipeline import TrainingConfig, run_training_pipeline
from storm_utils.config_paths import get_project_paths
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

paths = get_project_paths()
huxt_run_id = 3

# Phase 1.2: Ensemble spread exploration
config_phase2_architecture_taper = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    
    model_type='mlp',
    mlp_include_ensemble_spread=False,
    mlp_ensemble_percentiles=[5, 50, 95, ],
    mlp_n_epochs=200,
    mlp_learning_rate=0.005,
    mlp_patience=10,
    
    lead_times=[12],
    random_seeds=[42],
    test_folds=[0],
    storm_thresholds=[4.5],
    balance=True,
    save_results=True,
    
    omni_subset=[], # Don't use any omni to purely visualise the 

    experiment_phase='phase1_ensemble_spread',
)

In [ ]:
# Phase 2: Architecture experiments
architecture_configs = [
    # Small but capable
    ([300, 150, 75], 'baseline'),       # ~565k params
    ([256, 128, 64], 'taper_pow2'),     # ~242k params
    ([200, 100, 50], 'taper_small'),    # ~195k params
    ([150, 75, 40], 'taper_tiny'),      # ~135k params
    
    # Uniform (more regularization from structure)
    ([200, 200, 200], 'uniform_200'),   # ~234k params
    ([150, 150, 150], 'uniform_150'),   # ~155k params
    
    # Depth tests (keeping narrow)
    ([200, 100], 'depth_2'),            # ~193k params
    ([200, 150, 100, 50], 'depth_4'),   # ~228k params
    
    # Extreme compression (risky but interesting)
    ([100, 100, 100], 'uniform_100'),   # ~107k params
    ([400, 50, 400], 'bottleneck_extreme'), # ~373k params - force feature learning
]

# Base config with BEST settings from Phase 1
base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    
    model_type='mlp',
    mlp_ensemble_percentiles=[5, 16, 50, 84, 95], 
    mlp_include_ensemble_spread=False,  
    
    mlp_n_epochs=100, ## Let the models train as long as they need to to "converge"
    mlp_learning_rate=0.001,
    mlp_patience=10,
    
    lead_times=[12],
    random_seeds=[42],
    test_folds=[0],
    storm_thresholds=[4.5],
    balance=True,
    save_results=True,
    
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    experiment_phase='phase2_architecture',
)


for architecture, arch_name in architecture_configs:
    logger.info(f"\n{'='*80}")
    logger.info(f"Testing architecture: {arch_name}")
    logger.info(f"{'='*80}\n")
    
    config = copy.deepcopy(base_config)
    config.mlp_architecture = architecture
    config.model_name = f"MLP_{arch_name}"  # Optional: override base name
    
    try:
        run_training_pipeline(config)
    except Exception as e:
        logger.error(f"Failed {arch_name}: {e}")
        continue

logger.info("\nPhase 2 architecture experiments complete!")

## Phase 2 - Unbalanced training and testing set 

In [ ]:
# Setup Experiment and run training for MLP
from storm_regression.training_pipeline import TrainingConfig, run_training_pipeline
from storm_utils.config_paths import get_project_paths
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

paths = get_project_paths()
huxt_run_id = 3

# Phase 1.2: Ensemble spread exploration
config_phase2_architecture_taper = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    
    model_type='mlp',
    mlp_include_ensemble_spread=False,
    mlp_ensemble_percentiles=[5, 50, 95, ],
    mlp_n_epochs=200,
    mlp_learning_rate=0.005,
    mlp_patience=10,
    
    lead_times=[12],
    random_seeds=[42],
    test_folds=[0],
    storm_thresholds=[4.5],
    balance=True,
    save_results=True,
    
    omni_subset=[], # Don't use any omni to purely visualise the 

    experiment_phase='phase1_ensemble_spread',
)

In [ ]:
# Phase 2: Architecture experiments
architecture_configs = [
    # Small but capable
    ([300, 150, 75], 'baseline'),       # ~565k params
    ([256, 128, 64], 'taper_pow2'),     # ~242k params
    ([200, 100, 50], 'taper_small'),    # ~195k params
    ([150, 75, 40], 'taper_tiny'),      # ~135k params
    
    # Uniform (more regularization from structure)
    ([200, 200, 200], 'uniform_200'),   # ~234k params
    ([150, 150, 150], 'uniform_150'),   # ~155k params
    
    # Depth tests (keeping narrow)
    ([200, 100], 'depth_2'),            # ~193k params
    ([200, 150, 100, 50], 'depth_4'),   # ~228k params
    
    # Extreme compression (risky but interesting)
    ([100, 100, 100], 'uniform_100'),   # ~107k params
    ([400, 50, 400], 'bottleneck_extreme'), # ~373k params - force feature learning
]

# Base config with BEST settings from Phase 1
base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    
    model_type='mlp',
    mlp_ensemble_percentiles=[5, 16, 50, 84, 95], 
    mlp_include_ensemble_spread=False,  
    
    mlp_n_epochs=1000, ## Let the models train as long as they need to to "converge"
    mlp_learning_rate=0.005,
    mlp_patience=10,
    
    lead_times=[12],
    random_seeds=[42],
    test_folds=[0],
    storm_thresholds=[4.5],
    balance=False,
    save_results=True,
    
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    experiment_phase='phase2_architecture_unbalanced_storms',
)


for architecture, arch_name in architecture_configs:
    logger.info(f"\n{'='*80}")
    logger.info(f"Testing architecture: {arch_name}")
    logger.info(f"{'='*80}\n")
    
    config = copy.deepcopy(base_config)
    config.mlp_architecture = architecture
    config.model_name = f"MLP_{arch_name}"  # Optional: override base name
    
    try:
        run_training_pipeline(config)
    except Exception as e:
        logger.error(f"Failed {arch_name}: {e}")
        continue

logger.info("\nPhase 2 architecture experiments complete!")

In [ ]:
# Base config with BEST settings from Phase 1
from storm_regression.training_pipeline import NLLLossConfig

base_loss_config = NLLLossConfig()

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    
    model_type='mlp',
    mlp_ensemble_percentiles=[5, 16, 50, 84, 95], 
    mlp_include_ensemble_spread=False,  
    
    mlp_n_epochs=1000, ## Let the models train as long as they need to to "converge"
    mlp_learning_rate=0.005,
    mlp_patience=10,
    mlp_architecture=[150, 150, 150],

    loss_config = base_loss_config,
    
    lead_times=[12],
    random_seeds=[42],
    test_folds=[0],
    storm_thresholds=[4.5],
    balance=False,
    save_results=True,
    
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    experiment_phase='phase3_loss_functions_nll',
)

loss_configs = (
    (1, 1),
    (1, 0.5),
    (0.5, 1),
    (1, 0.33),
    (0.33, 1),
)

for w_sigma, w_accuracy in loss_configs:
    config = copy.deepcopy(base_config)
    
    config.loss_config.w_sigma = w_sigma
    config.loss_config.w_accuracy = w_accuracy
    
    try:
        run_training_pipeline(config)
    except Exception as e:
        logger.error(f"Failed {arch_name}: {e}")
        continue

logger.info("\nPhase 3 Loss Function experiments complete!")

In [ ]:
# Base config with BEST settings from Phase 1
from storm_regression.training_pipeline import NLLLossConfig
paths = get_project_paths()

base_loss_config = NLLLossConfig()
base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    
    model_type='mlp',
    mlp_ensemble_percentiles=[5, 16, 50, 84, 95], 
    mlp_include_ensemble_spread=False,  
    
    mlp_n_epochs=1000,
    mlp_learning_rate=0.005,
    mlp_patience=10,
    mlp_architecture=[150, 150, 150],
    loss_config=base_loss_config,
    
    lead_times=[12],
    random_seeds=[42],
    test_folds=[0],
    storm_thresholds=[4.5],
    balance=False,
    save_results=True,
    
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    experiment_phase='phase3_loss_functions_hp30_weighting',
)

intensity_types = [None, 'linear', 'quadratic', 'threshold']

for intensity_type in intensity_types:
    config = copy.deepcopy(base_config)
    config.loss_config.intensity_type = intensity_type
    config.experiment_phase = f"phase3_loss_functions_hp30_weighting"
    
    try:
        run_training_pipeline(config)
    except Exception as e:
        logger.error(f"Failed intensity_type={intensity_type}: {e}")
        continue

logger.info("\nPhase 3 Hp30 weighting experiments complete!")

## Phase 3 Loss Functions - Hp30 weighting

In [ ]:
# Base config with BEST settings from Phase 1
from storm_regression.training_pipeline import NLLLossConfig
paths = get_project_paths()

base_loss_config = NLLLossConfig()
base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / f'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',

    model_type='mlp',
    mlp_ensemble_percentiles=[5, 16, 50, 84, 95], 
    mlp_include_ensemble_spread=True,  
    
    mlp_n_epochs=1000,
    mlp_learning_rate=0.01,
    mlp_patience=10,
    mlp_architecture=[150, 150, 150],
    loss_config=base_loss_config,
    
    lead_times=[12],
    random_seeds=[42],
    test_folds=[0],
    balance=False,
    save_results=True,
    remove_cmes=True,
    
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    
    experiment_phase='phase3_loss_functions_hp30_weighting',
)

intensity_configs = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 50.0]

for scale_factor in intensity_configs:
    config = copy.deepcopy(base_config)
    config.loss_config.intensity_type = 'linear' if scale_factor is not None else None
    config.loss_config.intensity_strength = scale_factor if scale_factor is not None else 1.0
    config.experiment_phase = 'phase3_loss_functions_hp30_weighting_scale_linear'

    label = f"linear scale={scale_factor}"
    try:
        logger.info(f"Running: {label}")
        run_training_pipeline(config)
    except Exception as e:
        logger.error(f"Failed {label}: {e}")
        continue

logger.info("\nPhase 3 Hp30 weighting experiments complete!")

In [ ]:
## Phase 3 - Loss Functions CRPS Loss 

In [ ]:
from storm_regression.training_pipeline import NLLLossConfig, CRPSLossConfig, AsymmetricLossConfig
paths = get_project_paths()

# Configure CRPS Loss
crps_loss_config = CRPSLossConfig()

# Configure NLL Loss
nll_loss_config = NLLLossConfig()
nll_loss_config.intensity_type = 'linear' 
nll_loss_config.intensity_strength = 2.0

# Configure Asymmetric Loss
asym_loss_config = AsymmetricLossConfig()

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    model_type='mlp',
    mlp_ensemble_percentiles=[5, 16, 50, 84, 95],
    mlp_include_ensemble_spread=True,

    mlp_n_epochs=1000,
    mlp_learning_rate=0.01,
    mlp_patience=10,
    mlp_architecture=[150, 150, 150],

    lead_times=[12],
    random_seeds=[42],
    test_folds=[0],
    balance=False,
    save_results=True,
    remove_cmes=True,

    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    experiment_phase='phase3_loss_functions_variety',
)

try:
    for loss_config in [crps_loss_config, nll_loss_config, asym_loss_config]:
        config = copy.deepcopy(base_config)
        config.loss_config = loss_config
        logger.info(f"Running: {config.loss_config.loss_type} loss")
        run_training_pipeline(config)
except Exception as e:
    logger.error(f"Failed {config.loss_config.loss_type} loss: {e}")

logger.info("Phase 3 Loss Function Variety experiment complete!")

In [ ]:
from storm_regression.training_pipeline import CRPSLossConfig
paths = get_project_paths()

crps_loss_config = CRPSLossConfig()

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    model_type='mlp',
    mlp_ensemble_percentiles=[5, 16, 50, 84, 95],
    mlp_include_ensemble_spread=True,
    mlp_n_epochs=1000,
    mlp_learning_rate=0.010,
    mlp_patience=10,
    mlp_architecture=[150, 150, 150],
    loss_config=crps_loss_config,
    lead_times=[12],
    random_seeds=[42],
    test_folds=[0],
    n_ensembles=200,
    balance=False,
    save_results=True,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    experiment_phase='phase3_ensemble_selection',
)

# Each entry is:
# (filter_ensemble, n_ensemble_keep, ensemble_selection_method, mlp_ensemble_percentiles)
ensemble_configs = [
    # ── Baseline: no filtering, snap ─────────────────────────────────────
    # (False, 200, 'snap',         [5, 16, 50, 84, 95]),

    # ── Filtering with snap, variety of n_keep ───────────────────────────
    (True,  100, 'snap',         [5, 16, 50, 84, 95]),
    (True,   50, 'snap',         [5, 16, 50, 84, 95]),
    (True,   25, 'snap',         [5, 16, 50, 84, 95]),
    (True,   10, 'snap',         [5, 16, 50, 84, 95]),

    # ── Filtering with per_timestep, variety of n_keep ───────────────────
    (True,  100, 'per_timestep', [5, 16, 50, 84, 95]),
    (True,   50, 'per_timestep', [5, 16, 50, 84, 95]),
    (True,   25, 'per_timestep', [5, 16, 50, 84, 95]),
    (True,   10, 'per_timestep', [5, 16, 50, 84, 95]),

    # ── No filtering, per_timestep ────────────────────────────────────────
    (False, 200, 'per_timestep', [5, 16, 50, 84, 95]),

    # ── Wider percentile range ────────────────────────────────────────────
    (True,   50, 'snap',         [1, 5, 16, 50, 84, 95, 99]),
    (True,   50, 'per_timestep', [1, 5, 16, 50, 84, 95, 99]),

    # ── Narrower percentile range ─────────────────────────────────────────
    (True,   50, 'snap',         [25, 50, 75]),
    (True,   50, 'per_timestep', [25, 50, 75]),
]

for filter_ensemble, n_keep, selection_method, percentiles in ensemble_configs:
    config = copy.deepcopy(base_config)
    config.filter_ensemble             = filter_ensemble
    config.n_ensemble_keep             = n_keep
    config.ensemble_selection_method   = selection_method
    config.mlp_ensemble_percentiles    = percentiles

    label = (f"filter={filter_ensemble}, n_keep={n_keep}, "
             f"method={selection_method}, percentiles={percentiles}")

    perc_str = '-'.join(str(p) for p in percentiles)
    config.run_name = (
        f"filter{filter_ensemble}"
        f"_keep{n_keep}"
        f"_{selection_method}"
        f"_p{perc_str}"
    )

    try:
        logger.info(f"Running: {label}")
        run_training_pipeline(config)
    except Exception as e:
        logger.error(f"Failed {label}: {e}")
        continue

logger.info("Phase 3 ensemble selection experiments complete!")

In [ ]:
from storm_regression.training_pipeline import CRPSLossConfig
paths = get_project_paths()
crps_loss_config = CRPSLossConfig()

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    model_type='mlp',
    mlp_ensemble_percentiles=[5, 16, 50, 84, 95],
    mlp_include_ensemble_spread=True,
    mlp_n_epochs=1000,
    mlp_learning_rate=0.005,
    mlp_patience=10,
    mlp_architecture=[150, 150, 150],
    loss_config=crps_loss_config,
    lead_times=[12],
    random_seeds=[42],
    test_folds=[0],
    n_ensembles=200,
    balance=False,
    save_results=True,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    experiment_phase='phase3_ensemble_subsampling_with_raw_ensembles',
)

# Test subsampling with snap method across a wide range of n_keep values,
# from very aggressive filtering all the way up to the full ensemble.
# Also test a range of percentile sets to see how many percentile features helps.
ensemble_configs = [
    # ── n_keep sweep, standard percentiles ───────────────────────────────
    (True,    5, 'snap', [5, 16, 50, 84, 95]),
    (True,   10, 'snap', [5, 16, 50, 84, 95]),
    (True,   20, 'snap', [5, 16, 50, 84, 95]),
    (True,   25, 'snap', [5, 16, 50, 84, 95]),
    (True,   50, 'snap', [5, 16, 50, 84, 95]),
    (True,   75, 'snap', [5, 16, 50, 84, 95]),
    (True,  100, 'snap', [5, 16, 50, 84, 95]),
    (True,  150, 'snap', [5, 16, 50, 84, 95]),
    (False, 200, 'snap', [5, 16, 50, 84, 95]),  # full ensemble baseline

    # ── n_keep sweep, denser percentiles ─────────────────────────────────
    (True,   10, 'snap', [5, 10, 25, 50, 75, 90, 95]),
    (True,   25, 'snap', [5, 10, 25, 50, 75, 90, 95]),
    (True,   50, 'snap', [5, 10, 25, 50, 75, 90, 95]),
    (True,  100, 'snap', [5, 10, 25, 50, 75, 90, 95]),
    (False, 200, 'snap', [5, 10, 25, 50, 75, 90, 95]),

    # ── n_keep sweep, very dense percentiles ─────────────────────────────
    (True,   50, 'snap', [2, 5, 10, 16, 25, 50, 75, 84, 90, 95, 98]),
    (True,  100, 'snap', [2, 5, 10, 16, 25, 50, 75, 84, 90, 95, 98]),
    (False, 200, 'snap', [2, 5, 10, 16, 25, 50, 75, 84, 90, 95, 98]),

    # ── Just median (minimal features) ───────────────────────────────────
    (True,   50, 'snap', [50]),
    (False, 200, 'snap', [50]),
]

for filter_ensemble, n_keep, selection_method, percentiles in ensemble_configs:
    config = copy.deepcopy(base_config)
    config.filter_ensemble           = filter_ensemble
    config.n_ensemble_keep           = n_keep
    config.ensemble_selection_method = selection_method
    config.mlp_ensemble_percentiles  = percentiles

    perc_str = '-'.join(str(p) for p in percentiles)
    n_keep_str = str(n_keep) if filter_ensemble else 'full'
    config.run_name = f"keep{n_keep_str}_snap_p{perc_str}"

    label = f"filter={filter_ensemble}, n_keep={n_keep}, percentiles={percentiles}"
    try:
        logger.info(f"Running: {label}")
        run_training_pipeline(config)
    except Exception as e:
        logger.error(f"Failed {label}: {e}")
        continue

logger.info("Phase 3 ensemble subsampling experiments complete!")

In [ ]:
from storm_regression.training_pipeline import CRPSLossConfig
paths = get_project_paths()
crps_loss_config = CRPSLossConfig()

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    model_type='mlp',
    mlp_ensemble_percentiles=[5, 16, 50, 84, 95],
    mlp_include_ensemble_spread=True,
    mlp_n_epochs=1000,
    mlp_learning_rate=0.005,
    mlp_patience=10,
    mlp_architecture=[150, 150, 150],
    loss_config=crps_loss_config,
    lead_times=[12],
    random_seeds=[42],
    test_folds=[0],
    n_ensembles=200,
    balance=False,
    save_results=True,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    experiment_phase='phase3_ensemble_subsampling_per_timestep',
)

# Test subsampling with snap method across a wide range of n_keep values,
# from very aggressive filtering all the way up to the full ensemble.
# Also test a range of percentile sets to see how many percentile features helps.
ensemble_configs = [
    # ── n_keep sweep, standard percentiles ───────────────────────────────
    (True,    5, 'per_timestep', [5, 16, 50, 84, 95]),
    (True,   10, 'per_timestep', [5, 16, 50, 84, 95]),
    (True,   20, 'per_timestep', [5, 16, 50, 84, 95]),
    (True,   25, 'per_timestep', [5, 16, 50, 84, 95]),
    (True,   50, 'per_timestep', [5, 16, 50, 84, 95]),
    (True,   75, 'per_timestep', [5, 16, 50, 84, 95]),
    (True,  100, 'per_timestep', [5, 16, 50, 84, 95]),
    (True,  150, 'per_timestep', [5, 16, 50, 84, 95]),
    (False, 200, 'per_timestep', [5, 16, 50, 84, 95]),  # full ensemble baseline

    # ── n_keep sweep, denser percentiles ─────────────────────────────────
    (True,   10, 'per_timestep', [5, 10, 25, 50, 75, 90, 95]),
    (True,   25, 'per_timestep', [5, 10, 25, 50, 75, 90, 95]),
    (True,   50, 'per_timestep', [5, 10, 25, 50, 75, 90, 95]),
    (True,  100, 'per_timestep', [5, 10, 25, 50, 75, 90, 95]),
    (False, 200, 'per_timestep', [5, 10, 25, 50, 75, 90, 95]),

    # ── n_keep sweep, very dense percentiles ─────────────────────────────
    (True,   50, 'per_timestep', [2, 5, 10, 16, 25, 50, 75, 84, 90, 95, 98]),
    (True,  100, 'per_timestep', [2, 5, 10, 16, 25, 50, 75, 84, 90, 95, 98]),
    (False, 200, 'per_timestep', [2, 5, 10, 16, 25, 50, 75, 84, 90, 95, 98]),

    # ── Just median (minimal features) ───────────────────────────────────
    (True,   50, 'per_timestep', [50]),
    (False, 200, 'per_timestep', [50]),
]

for filter_ensemble, n_keep, selection_method, percentiles in ensemble_configs:
    config = copy.deepcopy(base_config)
    config.filter_ensemble           = filter_ensemble
    config.n_ensemble_keep           = n_keep
    config.ensemble_selection_method = selection_method
    config.mlp_ensemble_percentiles  = percentiles

    perc_str = '-'.join(str(p) for p in percentiles)
    n_keep_str = str(n_keep) if filter_ensemble else 'full'
    config.run_name = f"keep{n_keep_str}_snap_p{perc_str}"

    label = f"filter={filter_ensemble}, n_keep={n_keep}, percentiles={percentiles}"
    try:
        logger.info(f"Running: {label}")
        run_training_pipeline(config)
    except Exception as e:
        logger.error(f"Failed {label}: {e}")
        continue

logger.info("Phase 3 ensemble subsampling experiments complete!")

In [ ]:
from storm_regression.training_pipeline import CRPSLossConfig
paths = get_project_paths()
crps_loss_config = CRPSLossConfig()

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    model_type='mlp',
    mlp_ensemble_percentiles=[5, 16, 50, 84, 95],
    mlp_include_ensemble_spread=True,
    mlp_n_epochs=1000,
    mlp_learning_rate=0.005,
    mlp_patience=10,
    mlp_architecture=[150, 150, 150],
    loss_config=crps_loss_config,
    lead_times=[12],
    random_seeds=[42],
    test_folds=[0],
    n_ensembles=200,
    balance=False,
    save_results=True,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    experiment_phase='phase3_ensemble_subsampling_per_timestep_crps_loss',
)

# Test subsampling with snap method across a wide range of n_keep values,
# from very aggressive filtering all the way up to the full ensemble.
# Also test a range of percentile sets to see how many percentile features helps.
ensemble_configs = [
    # ── n_keep sweep, standard percentiles ───────────────────────────────
    (True,   20, 'per_timestep', [5, 16, 50, 84, 95]),
    (True,   50, 'per_timestep', [5, 16, 50, 84, 95]),
    (True,  100, 'per_timestep', [5, 16, 50, 84, 95]),
    (True,  150, 'per_timestep', [5, 16, 50, 84, 95]),
    (False, 200, 'per_timestep', [5, 16, 50, 84, 95]),  # full ensemble baseline

]

for filter_ensemble, n_keep, selection_method, percentiles in ensemble_configs:
    config = copy.deepcopy(base_config)
    config.filter_ensemble           = filter_ensemble
    config.n_ensemble_keep           = n_keep
    config.ensemble_selection_method = selection_method
    config.mlp_ensemble_percentiles  = percentiles

    perc_str = '-'.join(str(p) for p in percentiles)
    n_keep_str = str(n_keep) if filter_ensemble else 'full'
    config.run_name = f"keep{n_keep_str}_snap_p{perc_str}"

    label = f"filter={filter_ensemble}, n_keep={n_keep}, percentiles={percentiles}"
    try:
        logger.info(f"Running: {label}")
        run_training_pipeline(config)
    except Exception as e:
        logger.error(f"Failed {label}: {e}")
        continue

logger.info("Phase 3 ensemble subsampling experiments complete!")

## Phase 4: twCRPS

### Formulation testing

In [ ]:
from storm_regression.training_pipeline import tw_crps_lognormal_loss, crps_lognormal_loss
import torch

mu = torch.tensor([1.0, 1.5])
sigma = torch.tensor([0.4, 0.3])
y = torch.tensor([3.0, 5.0])

# (a) threshold below all data -> v is linear -> twCRPS == ordinary CRPS
torch.manual_seed(0); a = tw_crps_lognormal_loss(mu, sigma, y, threshold=-10, sharpness=1.0, n_samples=50000)
torch.manual_seed(0); b = crps_lognormal_loss(mu, sigma, y, n_samples=50000)   # your existing

passed = (float(a) - float(b)) < 1e-3
print(f"Test 1: {'passed' if passed else 'failed'}")
print(float(a), float(b), "\n")   # should match to ~1e-3

# (b) gradient flows
mu_ = mu.clone().requires_grad_(True)
tw_crps_lognormal_loss(mu_, sigma, y, threshold=4.66).backward()
print(f"Test 2: passed if tensor is finite and non-zero")
print(mu_.grad)             # finite, non-zero


In [ ]:
from storm_regression.training_pipeline import run_training_pipeline
from storm_utils.config_paths import get_project_paths

import logging
import copy
from storm_regression.training_pipeline import (
    TrainingConfig, NLLLossConfig, CRPSLossConfig, TwCRPSLossConfig
)

logger = logging.getLogger(__name__)
paths = get_project_paths()

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    model_type='mlp',
    mlp_ensemble_percentiles=[5, 16, 50, 84, 95],
    mlp_include_ensemble_spread=True,
    mlp_n_epochs=1000,
    mlp_learning_rate=0.005,
    mlp_patience=10,
    mlp_architecture=[150, 150, 150],
    lead_times=[12],
    random_seeds=[42],
    test_folds=[0],
    n_ensembles=200,
    filter_ensemble=False,
    ensemble_selection_method='snap',
    balance=False,
    save_results=True,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    loss_config=NLLLossConfig(),                 # overwritten below
)

# ONLY the loss varies. Intensity weighting OFF on nll/crps (the improper path twCRPS replaces).
loss_configs = [
    # ('nll',          NLLLossConfig(intensity_strength=0)),
    ('crps',         CRPSLossConfig(intensity_strength=0)),
    ('twcrps_t4.66', TwCRPSLossConfig(threshold=4.66, sharpness=2.0, n_samples=3000)),
    ('twcrps_t5.0',  TwCRPSLossConfig(threshold=5.0,  sharpness=2.0, n_samples=3000)),
    ('twcrps_t6.5',  TwCRPSLossConfig(threshold=6.5,  sharpness=2.0, n_samples=3000)),
]

for tag, lc in loss_configs:
    config = copy.deepcopy(base_config)
    config.loss_config = lc
    config.experiment_phase = f'phase4_loss_comparison'
    config.run_name = f'loss_{tag}'
    try:
        logger.info(f"Running loss = {tag}")
        run_training_pipeline(config)
    except Exception as e:
        logger.error(f"Failed loss={tag}: {e}")
        continue

logger.info("Phase 4 twCRPS loss comparison complete!")

In [ ]:
## Mixture Overfit Test

import torch
from torch.utils.data import Subset, DataLoader
from storm_utils.data_structure import ForecastingDataset
from storm_utils.config_paths import get_project_paths
from storm_regression.training_pipeline import train_mlp, MixtureNLLLossConfig

paths = get_project_paths()

# 1. Build dataset exactly as the pipeline does
dataset = ForecastingDataset(
    parquet_path=str(paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet'),
    discontinuity_path=str(paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy'),
    seed=42,
    Nens=200,
    lead_time_hours=12,
    forecast_duration_hours=24,
    stride_hours=24,
)

# 2. Match the pipeline's preprocessing (no balance, remove CMEs, set OMNI)
dataset.remove_cmes(inplace=True)
train_indices, test_indices = dataset.rotation_aligned_train_test_split(
    train_ratio=0.8, test_fold=0
)
dataset.set_omni_columns(['Bz_GSM', 'Dst', 'n_sw'])

# 3. Grab a TINY slice of the training set — deliberately include storms
#    (pick indices spread across the set so it's not all-quiet)
tiny_indices = train_indices[:50]                     # first 50 train windows
tiny = Subset(dataset, tiny_indices)
tiny_loader = DataLoader(tiny, batch_size=50, shuffle=True)

# 4. Overfit: high patience + many epochs so it can memorise
model, scaler = train_mlp(
    tiny_loader, n_ensembles=200,
    loss_config=MixtureNLLLossConfig(n_components=2),
    include_ensemble_spread=True,
    ensemble_percentiles=[5,16,50,84,95],
    ensemble_selection_method='snap', filter_ensemble=False,
    architecture=[150,150,150],
    n_epochs=500,
    learning_rate=0.0005,      # 10x lower — the key change
    patience=500,
    device='cpu', debug=False,
)

In [ ]:
from storm_regression.training_pipeline import TrainingConfig, MixtureNLLLossConfig, run_training_pipeline

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    model_type='mlp',
    mlp_ensemble_percentiles=[5, 16, 50, 84, 95],
    mlp_include_ensemble_spread=True,
    mlp_n_epochs=1000,
    mlp_learning_rate=0.001,        # lower than single-LogNormal's 0.005 — mixtures are LR-sensitive
    mlp_patience=20,                 # real early stopping (NOT the disabled patience from the overfit test)
    mlp_architecture=[150, 150, 150],
    lead_times=[12],
    random_seeds=[42],               # single run first to confirm it works end-to-end
    test_folds=[0],
    n_ensembles=200,
    filter_ensemble=False,
    ensemble_selection_method='snap',
    balance=False,
    save_results=True,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    loss_config=MixtureNLLLossConfig(n_components=2),
    experiment_phase='phase5_mixture_nll',
    run_name='mixture_K2_nll_highLR',
)

run_training_pipeline(base_config)

In [ ]:
# Confirmation that the Mixture is being trained properly

from pathlib import Path
from storm_regression.results_io import load_results
from storm_regression.predictive import forecast_from_results

# Usage example:
results_dir = paths['regression_results'] / 'HUXt3' / 'phase5_mixture_head_sanity_check'
results_path = results_dir / 'results_loss_mixture_nll.pkl'

results, config, _ = load_results(results_path)

mu  = np.asarray(results['mu_m'])      # (B, K)
sig = np.asarray(results['sigma_m'])   # (B, K)
w   = np.asarray(results['alpha'])     # (B, K)

# 1. Are the two components' means actually separated, per window?
mu_gap = np.abs(mu[:, 0] - mu[:, 1])             # (B,)
print(f"|mu_0 - mu_1|:   mean={mu_gap.mean():.3f}  median={np.median(mu_gap):.3f}  max={mu_gap.max():.3f}")

# 2. Are both components ever actually used? (weights not pinned to one)
print(f"weight on comp 0: mean={w[:,0].mean():.3f}  min={w[:,0].min():.3f}  max={w[:,0].max():.3f}")
frac_decisive = np.mean((w.max(axis=1) > 0.95))  # fraction of windows dominated by one comp
print(f"fraction of windows >95% one component: {frac_decisive:.2%}")

# 3. Do the components separate MORE on storm windows? (the thing we WANT)
storm = results['y_test'] > 4.66
print(f"mean |mu gap| quiet={mu_gap[~storm].mean():.3f}  storm={mu_gap[storm].mean():.3f}")

# 4. Verdict
if mu_gap.mean() < 0.05 and w[:,0].std() < 0.05:
    print(">>> LIKELY COLLAPSED: components are nearly identical and weights barely vary.")
else:
    print(">>> Components are distinct and/or weights adapt per window — not collapsed.")

## A quick LR and Patience test 

In [ ]:
import copy
from storm_regression.training_pipeline import TrainingConfig, MixtureNLLLossConfig, run_training_pipeline

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    model_type='mlp',
    mlp_ensemble_percentiles=[5, 16, 50, 84, 95],
    mlp_include_ensemble_spread=True,
    mlp_n_epochs=1000,
    mlp_architecture=[150, 150, 150],
    lead_times=[12],
    random_seeds=[42],
    test_folds=[0],
    n_ensembles=200,
    filter_ensemble=False,
    ensemble_selection_method='snap',
    balance=False,
    save_results=True,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    loss_config=MixtureNLLLossConfig(n_components=2),
    experiment_phase='phase5_mixture_nll',
)

# (learning_rate, patience) combinations
grid = [
    (0.0001, 20),
    (0.001,  20),
    (0.0001, 60),
    (0.001,  60),
]

for lr, patience in grid:
    config = copy.deepcopy(base_config)
    config.mlp_learning_rate = lr
    config.mlp_patience      = patience
    lr_tag = f"lr{lr:g}".replace('.', 'p')        # e.g. lr0p001
    pat_tag = f"pat{patience}"
    config.run_name = f"mixture_K2_nll_{lr_tag}_{pat_tag}"
    run_training_pipeline(config)

print("\nGrid complete.")

In [ ]:
## Phase 3a.4.1 - Validation based early stopping

import copy
from storm_regression.training_pipeline import TrainingConfig, MixtureNLLLossConfig, run_training_pipeline

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    model_type='mlp',
    mlp_ensemble_percentiles=[5, 16, 50, 84, 95],
    mlp_include_ensemble_spread=True,
    mlp_n_epochs=1000,
    mlp_architecture=[150, 150, 150],
    lead_times=[12],
    random_seeds=[42],
    test_folds=[0],
    n_ensembles=200,
    filter_ensemble=False,
    ensemble_selection_method='snap',
    balance=False,
    save_results=True,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    loss_config=MixtureNLLLossConfig(n_components=2),
    experiment_phase = 'phase5_mixture_nll_validation_early_stopping',
)

# (learning_rate, patience) combinations
grid = [
    (0.0001, 20),
    (0.001,  20),
    (0.0001, 60),
    (0.001,  60),
]

for lr, patience in grid:
    config = copy.deepcopy(base_config)
    config.mlp_learning_rate = lr
    config.mlp_patience      = patience
    lr_tag = f"lr{lr:g}".replace('.', 'p')        # e.g. lr0p001
    pat_tag = f"pat{patience}"
    config.run_name = f"mixture_K2_nll_{lr_tag}_{pat_tag}_valstop"
    run_training_pipeline(config)

print("\nGrid complete.")

In [ ]:
## Phase 3a.4.2 - Percentile representation search
import copy
from storm_regression.training_pipeline import TrainingConfig, MixtureNLLLossConfig, run_training_pipeline

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    model_type='mlp',
    mlp_include_ensemble_spread=True,
    mlp_n_epochs=1000,
    mlp_architecture=[150, 150, 150],
    mlp_learning_rate=0.0001,        # fixed at the best-generalising LR from the grid
    mlp_patience=20,                 # honest early stopping now does the work
    lead_times=[12],
    random_seeds=[42],
    test_folds=[0],
    n_ensembles=200,
    filter_ensemble=False,
    ensemble_selection_method='snap',
    balance=False,
    save_results=True,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    loss_config=MixtureNLLLossConfig(n_components=2),
    experiment_phase='phase5_mixture_nll_percentiles_search',
)

# (label, percentile list) — sparse -> dense, "full ensemble" approached via dense percentiles
percentile_sets = [
    ('p50',        [50]),                               # median only (floor / baseline)
    ('p5-95',      [5, 16, 50, 84, 95]),                # current default
    ('p10s',       list(range(10, 100, 10))),           # 9 percentiles (deciles)
    ('p5s',        list(range(5, 100, 5))),             # 19 percentiles
    ('p1s',        list(range(1, 100))),                # 99 percentiles (~full ensemble info)
]

for label, percentiles in percentile_sets:
    config = copy.deepcopy(base_config)
    config.mlp_ensemble_percentiles = percentiles
    config.run_name = f"mixture_K2_nll_perc_{label}_valstop"
    print(f"\n=== Running {config.run_name}  ({len(percentiles)} percentiles) ===")
    run_training_pipeline(config)

print("\nGrid complete.")

In [ ]:
## Phase 3a.5 - K sweep: does mixture flexibility fix the storm tail?
import copy
from storm_regression.training_pipeline import TrainingConfig, MixtureNLLLossConfig, NLLLossConfig, run_training_pipeline

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    model_type='mlp',
    mlp_ensemble_percentiles=[5, 16, 50, 84, 95],   # default representation, held fixed
    mlp_include_ensemble_spread=True,
    mlp_n_epochs=1000,
    mlp_architecture=[150, 150, 150],
    mlp_learning_rate=0.0001,        # best-generalising LR; held fixed across K
    mlp_patience=20,
    lead_times=[12],
    random_seeds=[42],
    test_folds=[0],
    n_ensembles=200,
    filter_ensemble=False,
    ensemble_selection_method='snap',
    balance=False,
    save_results=True,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    experiment_phase='phase5_mixture_nll_K_sweep',
)

K_values = [1, 2, 3, 5]

for K in K_values:
    config = copy.deepcopy(base_config)
    if K == 1:
        config.loss_config = NLLLossConfig() # single LogNormal model + single NLL
    else:
        config.loss_config = MixtureNLLLossConfig(n_components=K) 
    config.run_name = f"mixture_K{K}_nll_valstop"
    print(f"\n=== Running K={K} ===")
    run_training_pipeline(config)

print("\nK sweep complete.")

In [ ]:
## Phase 3b gate diagnostic - median-only vs full-percentile, 6h vs 12h
import copy
from storm_regression.training_pipeline import TrainingConfig, NLLLossConfig, run_training_pipeline

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    model_type='mlp',
    mlp_include_ensemble_spread=True,      # NOTE: overridden per-arm below
    mlp_n_epochs=1000,
    mlp_architecture=[150, 150, 150],
    mlp_learning_rate=0.0001,
    mlp_patience=20,
    random_seeds=[42],
    test_folds=[0],
    n_ensembles=200,
    filter_ensemble=False,
    ensemble_selection_method='snap',
    balance=False,
    save_results=True,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    loss_config=NLLLossConfig(),           # single LogNormal — isolates the feature question
    experiment_phase='phase5b_gate_diagnostic',
)

# arms: (label, percentiles, include_spread)
arms = [
    ('median_only', [50],                 False),   # ensemble reduced to its median, no spread
    ('full_perc',   [5, 16, 50, 84, 95],  True),    # full percentile spread + spread stats
]

for lead in [6, 12]:
    for label, percentiles, include_spread in arms:
        config = copy.deepcopy(base_config)
        config.lead_times = [lead]
        config.mlp_ensemble_percentiles = percentiles
        config.mlp_include_ensemble_spread = include_spread
        config.run_name = f"gate_{label}_lt{lead}"
        print(f"\n=== lead={lead}h | {label} ({len(percentiles)} perc, spread={include_spread}) ===")
        run_training_pipeline(config)

print("\nGate diagnostic runs complete.")

In [ ]:
#########################
# CONTINUED DIAGNOSTICS #
#########################

## Phase 3b gate diagnostic - median-only vs full-percentile, 6h vs 12h
import copy
from storm_regression.training_pipeline import TrainingConfig, NLLLossConfig, run_training_pipeline

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    model_type='mlp',
    mlp_include_ensemble_spread=True,      # NOTE: overridden per-arm below
    mlp_n_epochs=1000,
    mlp_architecture=[150, 150, 150],
    mlp_learning_rate=0.0001,
    mlp_patience=20,
    random_seeds=[42],
    test_folds=[0],
    n_ensembles=200,
    filter_ensemble=False,
    ensemble_selection_method='snap',
    balance=False,
    save_results=True,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    loss_config=NLLLossConfig(),           # single LogNormal — isolates the feature question
    experiment_phase='phase5b_gate_diagnostic',
)

# arms: (label, percentiles, include_spread)
arms = [
    ('p5s',   list(range(5, 100, 5)),  True),     # 19 percentiles
    ('p1s',   list(range(1, 100)),     True),        # 99 percentiles ~ full density
]

for lead in [6, 12]:
    for label, percentiles, include_spread in arms:
        config = copy.deepcopy(base_config)
        config.lead_times = [lead]
        config.mlp_ensemble_percentiles = percentiles
        config.mlp_include_ensemble_spread = include_spread
        config.run_name = f"gate_{label}_lt{lead}"
        print(f"\n=== lead={lead}h | {label} ({len(percentiles)} perc, spread={include_spread}) ===")
        run_training_pipeline(config)

print("\nGate diagnostic runs complete.")

## New Train/Val/Test splitting per CR

In [ ]:
#########################
# CONTINUED DIAGNOSTICS #
#########################

## Phase 3b gate diagnostic - median-only vs full-percentile, 6h vs 12h
import copy
from storm_regression.training_pipeline import TrainingConfig, NLLLossConfig, run_training_pipeline

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    model_type='mlp',
    mlp_include_ensemble_spread=True,
    mlp_n_epochs=1000,
    mlp_architecture=[150, 150, 150],
    mlp_learning_rate=0.0001,
    mlp_patience=20,
    random_seeds=[42],
    test_folds=[0],
    n_ensembles=200,
    filter_ensemble=False,
    ensemble_selection_method='snap',
    balance=False,
    save_results=True,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    loss_config=NLLLossConfig(),           # single LogNormal — isolates the feature question
    experiment_phase='phase5b_new_CR_split_test',
)

# arms: (label, percentiles, include_spread)
arms = [
    ('p5s',   list(range(5, 100, 5)),  True),     
]

for fold in range(5):
    fold = [fold,]
    for label, percentiles, include_spread in arms:
        config = copy.deepcopy(base_config)
        config.lead_times = [12]
        config.test_folds = fold
        config.mlp_ensemble_percentiles = percentiles
        config.mlp_include_ensemble_spread = include_spread
        config.run_name = f"gate_{label}_lt{lead}"
        print(f"\n=== lead={lead}h | {label} ({len(percentiles)} perc, spread={include_spread}) ===")
        run_training_pipeline(config)

print("\nGate diagnostic runs complete.")

## Final MLP tests before the auto-encoder is built

In [ ]:
#########################
# PHASE 1 — FEATURE / ENSEMBLE REPRESENTATION
# Fixed: arch [150,150,150], K=1 (NLL), lead 12h, fold 0, seed 42.
# One axis varied at a time from a baseline. Single fold/seed = fast relative comparison.
#########################
import copy
from storm_regression.training_pipeline import TrainingConfig, NLLLossConfig, run_training_pipeline

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    model_type='mlp',
    mlp_n_epochs=1000,
    mlp_architecture=[150, 150, 150],
    mlp_learning_rate=0.0001,
    mlp_patience=20,
    random_seeds=[42],
    test_folds=[0],
    lead_times=[12],                       # fixed during exploration
    n_ensembles=200,
    filter_ensemble=False,
    ensemble_selection_method='snap',
    balance=False,
    save_results=True,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    loss_config=NLLLossConfig(),           # K=1 isolates the feature question
    experiment_phase='phase5a_features',
)

P5S = list(range(5, 100, 5))               # 19 percentiles (baseline density)
P1S = list(range(1, 100))                  # 99 percentiles

# --- 1A: percentile density (spread ON, OMNI = all three) ---
density_arms = [
    ('median',  [50]),
    ('sparse',  [5, 25, 50, 75, 95]),
    ('p5s',     P5S),
    ('p1s',     P1S),
]
for label, pctls in density_arms:
    c = copy.deepcopy(base_config)
    c.mlp_ensemble_percentiles = pctls
    c.mlp_include_ensemble_spread = True
    c.run_name = f"p1_density_{label}"
    print(f"\n=== 1A density={label} ({len(pctls)} perc) ===")
    run_training_pipeline(c)

# --- 1B: spread flag (density = p5s; flip to your 1A winner after inspecting) ---
for spread in [True, False]:
    c = copy.deepcopy(base_config)
    c.mlp_ensemble_percentiles = P5S
    c.mlp_include_ensemble_spread = spread
    c.run_name = f"p1_spread_{'on' if spread else 'off'}"
    print(f"\n=== 1B spread={spread} ===")
    run_training_pipeline(c)

# --- 1C: OMNI ablation (§5.4) — density=p5s, spread ON ---
omni_arms = [
    ('none',   []),
    ('bz',     ['Bz_GSM']),
    ('dst',    ['Dst']),
    ('nsw',    ['n_sw']),
    ('all',    ['Bz_GSM', 'Dst', 'n_sw']),
]
for label, subset in omni_arms:
    c = copy.deepcopy(base_config)
    c.mlp_ensemble_percentiles = P5S
    c.mlp_include_ensemble_spread = True
    c.omni_subset = subset
    c.run_name = f"p1_omni_{label}"
    print(f"\n=== 1C omni={label} ({subset}) ===")
    run_training_pipeline(c)

print("\nPhase 1 complete.")

In [ ]:
#########################
# PHASE 1 (FOLD-ROBUST) — FEATURE / ENSEMBLE REPRESENTATION ACROSS ALL FOLDS
# Same matrix as Phase 1, now across all 5 folds (seed 42) to confirm the
# "median wins / detail hurts" finding is robust, not a fold-0 artefact.
# Fixed: arch [150,150,150], K=1 (NLL), lead 12h.
#########################
import copy
from storm_regression.training_pipeline import TrainingConfig, NLLLossConfig, run_training_pipeline

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    model_type='mlp',
    mlp_n_epochs=1000,
    mlp_architecture=[150, 150, 150],
    mlp_learning_rate=0.0001,
    mlp_patience=20,
    random_seeds=[42],
    test_folds=[0, 1, 2, 3, 4],            # <-- all folds now
    lead_times=[12],
    n_ensembles=200,
    filter_ensemble=False,
    ensemble_selection_method='snap',
    balance=False,
    save_results=True,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    loss_config=NLLLossConfig(),
    experiment_phase='phase5a_features_allfolds',
)

P5S = list(range(5, 100, 5))
P1S = list(range(1, 100))

# --- 1A: percentile density (spread ON, OMNI = all) ---
density_arms = [
    ('median',  [50]),
    ('sparse',  [5, 25, 50, 75, 95]),
    ('p5s',     P5S),
    ('p1s',     P1S),
]
for label, pctls in density_arms:
    c = copy.deepcopy(base_config)
    c.mlp_ensemble_percentiles = pctls
    c.mlp_include_ensemble_spread = True
    c.run_name = f"p1f_density_{label}"
    print(f"\n=== 1A density={label} ({len(pctls)} perc) — all folds ===")
    run_training_pipeline(c)

# --- 1B: spread flag (density = p5s) ---
for spread in [True, False]:
    c = copy.deepcopy(base_config)
    c.mlp_ensemble_percentiles = P5S
    c.mlp_include_ensemble_spread = spread
    c.run_name = f"p1f_spread_{'on' if spread else 'off'}"
    print(f"\n=== 1B spread={spread} — all folds ===")
    run_training_pipeline(c)

# --- 1C: OMNI ablation (§5.4) — density=p5s, spread ON ---
omni_arms = [
    ('none',   []),
    ('bz',     ['Bz_GSM']),
    ('dst',    ['Dst']),
    ('nsw',    ['n_sw']),
    ('all',    ['Bz_GSM', 'Dst', 'n_sw']),
]
for label, subset in omni_arms:
    c = copy.deepcopy(base_config)
    c.mlp_ensemble_percentiles = P5S
    c.mlp_include_ensemble_spread = True
    c.omni_subset = subset
    c.run_name = f"p1f_omni_{label}"
    print(f"\n=== 1C omni={label} ({subset}) — all folds ===")
    run_training_pipeline(c)

print("\nPhase 1 (fold-robust) complete.")

In [ ]:
#########################
# PHASE 2 — ARCHITECTURE
# Fixed: K=1 (NLL), lead 12h, seed 42. Explore on fold 0.
# Swept on BOTH median and p5s inputs, to test whether any architecture
# lets the richer representation overtake the median (Phase 1 showed it can't
# at [150,150,150]). Learning rate / patience folded in as secondary arms.
#########################
import copy
from storm_regression.training_pipeline import TrainingConfig, NLLLossConfig, run_training_pipeline

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    model_type='mlp',
    mlp_n_epochs=1000,
    mlp_learning_rate=0.0001,
    mlp_patience=20,
    random_seeds=[42],
    test_folds=[4],
    lead_times=[12],
    n_ensembles=200,
    filter_ensemble=False,
    ensemble_selection_method='snap',
    balance=False,
    save_results=True,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],   # OMNI-all (Phase 1: best non-median tail rep)
    loss_config=NLLLossConfig(),
    experiment_phase='phase5a_architecture',
)

P5S = list(range(5, 100, 5))

# architecture arms: (label, layers)
arch_arms = [
    ('shallow_narrow', [64, 64]),
    ('shallow_wide',   [256, 256]),
    ('baseline',       [150, 150, 150]),      # Phase 1 architecture, for continuity
    ('deep_narrow',    [64, 64, 64, 64]),
    ('deep_wide',      [256, 256, 256]),
    ('funnel',         [256, 128, 64]),
]

# run each architecture on BOTH inputs — the key comparison
input_arms = [
    ('median', [50],  False),                 # median, spread off (Phase 1 winner)
    ('p5s',    P5S,   True),                   # rich rep, spread on
]

for arch_label, layers in arch_arms:
    for in_label, pctls, spread in input_arms:
        c = copy.deepcopy(base_config)
        c.mlp_architecture = layers
        c.mlp_ensemble_percentiles = pctls
        c.mlp_include_ensemble_spread = spread
        c.run_name = f"{arch_label}_{in_label}"
        print(f"\n=== arch={arch_label} {layers} | input={in_label} ===")
        run_training_pipeline(c)

print("\nPhase 2 architecture sweep complete.")

In [ ]:
#########################
# PHASE 2 — FOLD VALIDATION of the architecture winner(s)
# Validate deep_narrow vs deep_wide (vs baseline) on the MEDIAN input,
# across all 5 folds, to lock the architecture. K=1 (NLL), lead 12h, seed 42.
# Median input, spread OFF (Phase 1: spread adds nothing).
#########################
import copy
from storm_regression.training_pipeline import TrainingConfig, NLLLossConfig, run_training_pipeline

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    model_type='mlp',
    mlp_n_epochs=1000,
    mlp_learning_rate=0.0001,
    mlp_patience=20,
    random_seeds=[42],
    test_folds=[0, 1, 2, 3, 4],            # all folds now — validation
    lead_times=[12],
    n_ensembles=200,
    filter_ensemble=False,
    ensemble_selection_method='snap',
    balance=False,
    save_results=True,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    loss_config=NLLLossConfig(),           # K=1
    experiment_phase='phase5a_architecture_foldvalidation',
)

# architectures to validate (median input, spread off)
arch_arms = [
    ('deep_narrow', [64, 64, 64, 64]),
    ('deep_wide',   [256, 256, 256]),
]

for arch_label, layers in arch_arms:
    c = copy.deepcopy(base_config)
    c.mlp_architecture = layers
    c.mlp_ensemble_percentiles = [50]      # median
    c.mlp_include_ensemble_spread = False  # spread adds nothing (Phase 1)
    c.run_name = f"p2val_{arch_label}_median"
    print(f"\n=== validating arch={arch_label} {layers} | median | all folds ===")
    run_training_pipeline(c)

print("\nPhase 2 fold-validation complete.")

In [ ]:
#########################
# PHASE 3 — K-SWEEP (multimodality confirmation) on locked architecture
# deep_narrow [64,64,64,64], median input, spread OFF, lead 12h, seed 42, all folds.
# K=1 (single LogNormal) vs K=2 vs K=3 mixture. Watch STORM-subset PIT-KS.
#########################
import copy
from storm_regression.training_pipeline import (
    TrainingConfig, NLLLossConfig, MixtureNLLLossConfig, run_training_pipeline
)

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    model_type='mlp',
    mlp_n_epochs=1000,
    mlp_architecture=[64, 64, 64, 64],     # LOCKED: deep_narrow
    mlp_learning_rate=0.0001,
    mlp_patience=20,
    random_seeds=[42],
    test_folds=[0, 1, 2, 3, 4],
    lead_times=[12],
    n_ensembles=200,
    filter_ensemble=False,
    ensemble_selection_method='snap',
    balance=False,
    save_results=True,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    mlp_ensemble_percentiles=[50],         # median
    mlp_include_ensemble_spread=False,     # spread adds nothing (Phase 1)
    experiment_phase='phase5a_ksweep',
)

# K arms: K=1 uses single-LogNormal NLL; K>=2 uses mixture NLL.
# VERIFY the mixture loss config class name / n_components field against your code.
k_arms = [
    ('K1', NLLLossConfig()),                              # single LogNormal
    ('K2', MixtureNLLLossConfig(n_components=2)),         # 2-component mixture
    ('K3', MixtureNLLLossConfig(n_components=3)),         # 3-component mixture
]

for label, loss_cfg in k_arms:
    c = copy.deepcopy(base_config)
    c.loss_config = loss_cfg
    c.run_name = f"{label}_deepnarrow_median"
    print(f"\n=== {label} | {loss_cfg.loss_type} | deep_narrow median | all folds ===")
    run_training_pipeline(c)

print("\nPhase 3a K-sweep complete.")

In [ ]:
#########################
# PHASE 3b — INTENSITY WEIGHTING DIAGNOSTIC
# Does up-weighting storms in the loss fix the undershoot, and if so does it
# improve SKILL (optimisational) or just shift the BIAS (informational ceiling)?
# K=2 deep_narrow median, lead 12h, seed 42, fold 0. step weighting.
#########################
import copy
from storm_regression.training_pipeline import (
    TrainingConfig, MixtureNLLLossConfig, run_training_pipeline
)

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    model_type='mlp',
    mlp_n_epochs=1000,
    mlp_architecture=[64, 64, 64, 64],     # deep_narrow (locked)
    mlp_learning_rate=0.0001,
    mlp_patience=20,
    random_seeds=[42],
    test_folds=[0],
    lead_times=[12],
    n_ensembles=200,
    filter_ensemble=False,
    ensemble_selection_method='snap',
    balance=False,
    save_results=True,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    mlp_ensemble_percentiles=[50],
    mlp_include_ensemble_spread=False,
    experiment_phase='phase5a_mixture_intensity_weighting',
)

for strength in [1, 2, 5, 10]:              # 1 = unweighted baseline
    c = copy.deepcopy(base_config)
    loss = MixtureNLLLossConfig(n_components=2)
    loss.intensity_type = 'step'
    loss.intensity_strength = strength
    loss.storm_threshold = 4.5
    c.loss_config = loss
    c.run_name = f"intensity_step{strength}"
    print(f"\n=== intensity step strength={strength} | K=2 deep_narrow median ===")
    run_training_pipeline(c)

print("\nPhase 3b intensity-weighting diagnostic complete.")

In [ ]:
#########################
# PHASE 4 — FINAL VALIDATION (flat-MLP control), multi-lead
# Locked config: K=2 mixture, deep_narrow, median input, spread off,
# UNWEIGHTED loss. 5 seeds x 5 folds x 3 leads {6,12,24} = 75 runs.
#########################
from storm_regression.training_pipeline import (
    TrainingConfig, MixtureNLLLossConfig, run_training_pipeline
)

loss = MixtureNLLLossConfig(n_components=2)
loss.intensity_type = 'none'                 # <-- UNWEIGHTED (diagnostic: weighting degrades overall)

config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    model_type='mlp',
    mlp_n_epochs=1000,
    mlp_architecture=[64, 64, 64, 64],       # deep_narrow (locked)
    mlp_learning_rate=0.0001,
    mlp_patience=20,
    random_seeds=[42, 1, 100, 12345, 151201],
    test_folds=[0, 1, 2, 3, 4],
    lead_times=[6, 12, 24],                  # <-- multi-lead
    n_ensembles=200,
    filter_ensemble=False,
    ensemble_selection_method='snap',
    balance=False,
    save_results=True,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    mlp_ensemble_percentiles=[50],
    mlp_include_ensemble_spread=False,
    loss_config=loss,
    experiment_phase='phase5a_final_control',
    run_name='control_k2_deepnarrow_median',
)

run_training_pipeline(config)
print("\nPhase 4 final validation complete.")

# Set Encoder

In [ ]:
from storm_regression.training_pipeline import (
    TrainingConfig, MixtureNLLLossConfig, run_training_pipeline
)

# Loss: IDENTICAL to the K=2 control — same mixture NLL, unweighted.
# This must match the control exactly so the only variable is the trunk.
loss = MixtureNLLLossConfig(n_components=2)
loss.intensity_type = 'none'                 # unweighted (matches control)

base_config = TrainingConfig(
    # ---- data / preprocessing: ALL IDENTICAL to the control ----
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    n_ensembles=200,
    filter_ensemble=False,
    ensemble_selection_method='snap',        # (unused by set encoder, but harmless)
    balance=False,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],   # same OMNI context (goes in post-pooling)

    # ---- CHANGED: model type ----
    model_type='set_encoder',                # <-- was 'mlp'

    # ---- NEW: set-encoder architecture fields ----
    se_phi_hidden=(64, 64),                  # per-member encoder phi
    se_rho_hidden=(64, 64),                  # decoder rho
    se_pool='mean',                          # <-- mean-pool FIRST (simplest)
    se_attention=False,                      # <-- attention OFF first (add later for §5.3)

    # ---- training hyperparams: carried over from the flat MLP ----
    mlp_n_epochs=1000,
    mlp_learning_rate=0.0001,
    mlp_patience=20,
    mlp_device='cpu',                        # or 'cuda'/'mps' — whatever you've been using
    batch_size=128,            # <-- CONFIRM: match whatever batch_size the control used

    # ---- NOT USED by the set encoder (percentile/flat-MLP-only) ----
    # mlp_architecture, mlp_ensemble_percentiles, mlp_include_ensemble_spread
    # are irrelevant here — the set encoder ignores them. Leave unset / at defaults.

    # ---- exploration scope: single fold/seed at 12h (fast first pass) ----
    random_seeds=[42],
    test_folds=[0],
    lead_times=[12],

    # ---- bookkeeping ----
    loss_config=loss,
    save_results=True,
    experiment_phase='phase5b_setencoder',
    run_name='p5b_meanpool_k2_vonly',
)

run_training_pipeline(base_config)
print("\nSet encoder (mean-pool, K=2, v-only) — first pass complete.")

In [ ]:
#########################
# PHASE 5b — POOLING SWEEP (max, meanmax)
# Does capturing EXTREME members (max) help storms vs the averaged member (mean)?
# K=2, v-only, 12h, fold 0, seed 42 — same as the mean-pool run, only se_pool changes.
#########################
import copy
from storm_regression.training_pipeline import (
    TrainingConfig, MixtureNLLLossConfig, run_training_pipeline
)

loss = MixtureNLLLossConfig(n_components=2)
loss.intensity_type = 'none'

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    n_ensembles=200,
    filter_ensemble=False,
    ensemble_selection_method='snap',
    balance=False,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    model_type='set_encoder',
    se_phi_hidden=(64, 64),
    se_rho_hidden=(64, 64),
    se_attention=False,
    mlp_n_epochs=1000,
    mlp_learning_rate=0.0001,
    mlp_patience=20,
    mlp_device='cpu',                  # match what you used for the mean run
    batch_size=128,
    random_seeds=[42],
    test_folds=[0],
    lead_times=[12],
    loss_config=loss,
    save_results=True,
    experiment_phase='phase5b_setencoder',
)

for pool in ['max', 'meanmax']:
    c = copy.deepcopy(base_config)
    c.se_pool = pool
    c.run_name = f"p5b_{pool}_k2_vonly"
    print(f"\n=== set encoder pool={pool} | K=2 v-only | 12h fold0 seed42 ===")
    run_training_pipeline(c)

print("\nPooling sweep complete.")

In [ ]:
#########################
# PHASE 5b — K sweep (1, 2, 3, 4, 5, 10)
# Do we see better separation of components with more components?
#########################
import copy
from storm_regression.training_pipeline import (
    TrainingConfig, MixtureNLLLossConfig, run_training_pipeline
)

base_loss_config = MixtureNLLLossConfig(n_components=3)
base_loss_config.intensity_type = 'none'

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    n_ensembles=200,
    filter_ensemble=False,
    ensemble_selection_method='snap',
    balance=False,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    model_type='set_encoder',
    se_phi_hidden=(64, 64),
    se_rho_hidden=(64, 64),
    se_attention=False,
    mlp_n_epochs=1000,
    mlp_learning_rate=0.0001,
    mlp_patience=20,
    mlp_device='cpu',  
    se_pool='mean',
    batch_size=128,
    random_seeds=[42],
    test_folds=[0],
    lead_times=[12],
    loss_config=loss,
    save_results=True,
    experiment_phase='phase5b_setencoder_ksweep',
)

for k in [1, 2, 3, 4, 5, 10]:
    c = copy.deepcopy(base_config)
    loss_config = copy.deepcopy(base_loss_config)
    loss_config.n_components = k
    c.loss_config = loss_config
    c.run_name = f"p5b_k{k}_vonly"
    print(f"\n=== set encoder mean pooling | K={k} v-only | 12h fold0 seed42 ===")
    run_training_pipeline(c)

print("\nPooling sweep complete.")

In [ ]:
#########################
# PHASE 5b — HOT-START DIAGNOSTIC (K=3 mean-pool)
# Does spreading the component medians at init let a high-Hp30 component survive,
# or does training pull it back down / bleed its weight (data doesn't want it)?
# A/B: init_spread False (default) vs True. Same everything else.
# K=3, v-only, 12h, fold 0, seed 42.
#########################
import copy
from storm_regression.training_pipeline import (
    TrainingConfig, MixtureNLLLossConfig, run_training_pipeline
)

loss = MixtureNLLLossConfig(n_components=3)
loss.intensity_type = 'none'

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    n_ensembles=200,
    filter_ensemble=False,
    ensemble_selection_method='snap',
    balance=False,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    model_type='set_encoder',
    se_phi_hidden=(64, 64),
    se_rho_hidden=(64, 64),
    se_pool='mean',
    se_attention=False,
    se_init_spread_range=(1.0, 7.0),       # spread across Hp30 1..7 when enabled
    mlp_n_epochs=1000,
    mlp_learning_rate=0.0001,
    mlp_patience=20,
    mlp_device='cpu',
    batch_size=128,
    random_seeds=[42],
    test_folds=[0],
    lead_times=[12],
    save_results=True,
    experiment_phase='phase5b_hotstart',
)

for init_spread in [False, True]:
    c = copy.deepcopy(base_config)
    c.loss_config = copy.deepcopy(loss)                 # ensure K=3 actually attaches
    c.se_init_spread = init_spread
    tag = 'hotstart' if init_spread else 'default'
    c.run_name = f"p5b_k3_meanpool_{tag}"
    print(f"\n=== K=3 mean-pool | init_spread={init_spread} ({tag}) | 12h fold0 seed42 ===")
    print(f"    confirm: n_components={c.loss_config.n_components}, "
          f"init_spread={c.se_init_spread}")     # guard against the K-propagation bug
    run_training_pipeline(c)

print("\nHot-start A/B complete.")

In [ ]:
#########################
# PHASE 5b — HOT-START RANGE SWEEP (K=3 mean-pool)
# Hot-start ON across a variety of initialisation ranges — does WHERE we place
# the components at init change the learned structure, and does a wider/higher
# range let a high-Hp30 component survive with more weight?
# K=3, v-only, 12h, fold 0, seed 42. init_spread=True throughout.
#########################
import copy
from storm_regression.training_pipeline import (
    TrainingConfig, MixtureNLLLossConfig, run_training_pipeline
)

loss = MixtureNLLLossConfig(n_components=3)
loss.intensity_type = 'none'

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    n_ensembles=200,
    filter_ensemble=False,
    ensemble_selection_method='snap',
    balance=False,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    model_type='set_encoder',
    se_phi_hidden=(64, 64),
    se_rho_hidden=(64, 64),
    se_pool='mean',
    se_attention=False,
    se_init_spread=True,                   # hot-start ON for all arms
    mlp_n_epochs=1000,
    mlp_learning_rate=0.0001,
    mlp_patience=20,
    mlp_device='cpu',
    batch_size=128,
    random_seeds=[42],
    test_folds=[0],
    lead_times=[12],
    save_results=True,
    experiment_phase='phase5b_hotstart_ranges',
)

# ranges to sweep — (low, high) component-median spread in Hp30 space
ranges = [
    (1.0, 5.0),     # conservative — top component only mid-range
    (1.0, 7.0),     # the run that worked
    (1.0, 9.0),     # push the top component further into the tail
    (2.0, 8.0),     # lift the whole spread up (no very-low component)
    (0.5, 9.0),     # widest — full plausible Hp30 span
]

for lo, hi in ranges:
    c = copy.deepcopy(base_config)
    c.loss_config = copy.deepcopy(loss)                # ensure K=3 attaches
    c.se_init_spread_range = (lo, hi)
    tag = f"{lo:g}_{hi:g}".replace('.', 'p')           # e.g. "1_7", "0p5_9"
    c.run_name = f"p5b_k3_hotstart_{tag}"
    print(f"\n=== K=3 mean-pool | hot-start range [{lo}, {hi}] | 12h fold0 seed42 ===")
    print(f"    confirm: n_components={c.loss_config.n_components}, "
          f"init_spread={c.se_init_spread}, range={c.se_init_spread_range}")
    run_training_pipeline(c)

print("\nHot-start range sweep complete.")

In [ ]:
#########################
# PHASE 5b — HOT-START RANGE SWEEP (K=3 mean-pool)
# Hot-start ON across a variety of initialisation ranges — does WHERE we place
# the components at init change the learned structure, and does a wider/higher
# range let a high-Hp30 component survive with more weight?
# K=3, v-only, 12h, fold 0, seed 42. init_spread=True throughout.
#########################
import copy
from storm_regression.training_pipeline import (
    TrainingConfig, MixtureNLLLossConfig, run_training_pipeline
)

base_loss = MixtureNLLLossConfig(n_components=2)
base_loss.intensity_type = 'none'

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    n_ensembles=200,
    filter_ensemble=False,
    ensemble_selection_method='snap',
    balance=False,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    model_type='set_encoder',
    se_phi_hidden=(64, 64),
    se_rho_hidden=(64, 64),
    se_pool='mean',
    se_attention=False,
    se_init_spread=True,                   # hot-start ON for all arms
    mlp_n_epochs=1000,
    mlp_learning_rate=0.0001,
    mlp_patience=20,
    mlp_device='cpu',
    batch_size=128,
    random_seeds=[42],
    test_folds=[0],
    lead_times=[12],
    save_results=True,
    experiment_phase='phase5b_hotstart_ranges',
)

# ranges to sweep — (low, high) component-median spread in Hp30 space
ranges = [
    (1.0, 5.0),     # conservative — top component only mid-range
    (1.0, 7.0),     # the run that worked
    (1.0, 9.0),     # push the top component further into the tail
    (2.0, 8.0),     # lift the whole spread up (no very-low component)
    (0.5, 9.0),     # widest — full plausible Hp30 span
    (1.0, 15.0),    # unrealistic big disparity
]

for lo, hi in ranges:
    for k in [2, 3, 4, 5]:
        c = copy.deepcopy(base_config)
        loss = copy.deepcopy(base_loss)
        loss.n_components = k
        c.loss_config = loss                                    # ensure k attaches
        c.se_init_spread_range = (lo, hi)
        tag = f"k{k}_{lo:g}_{hi:g}".replace('.', 'p')           # e.g. "1_7", "0p5_9"
        c.run_name = f"p5b_k3_hotstart_{tag}"
        print(f"\n=== K={k} mean-pool | hot-start range [{lo}, {hi}] | 12h fold0 seed42 ===")
        print(f"    confirm: n_components={c.loss_config.n_components}, "
              f"init_spread={c.se_init_spread}, range={c.se_init_spread_range}")
        run_training_pipeline(c)

print("\nHot-start range sweep complete.")

In [ ]:
## Hot start WITH storm weighted NLL Loss

#########################
# PHASE 5b — HOT-START + INTENSITY-WEIGHTING (K=3 mean-pool)
# Hypothesis: hot-start PLACES a component in the storm range; storm-weighted
# NLL then REWARDS keeping weight on it -> a genuine storm mode appears, instead
# of the weighting merely shifting the bulk (as it did with no storm component
# to absorb the emphasis).
# K=3, v-only, 12h, fold 0, seed 42. Hot-start [1,9] fixed; sweep weight strength.
#########################
import copy
from storm_regression.training_pipeline import (
    TrainingConfig, MixtureNLLLossConfig, run_training_pipeline
)

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    n_ensembles=200,
    filter_ensemble=False,
    ensemble_selection_method='snap',
    balance=False,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    model_type='set_encoder',
    se_phi_hidden=(64, 64),
    se_rho_hidden=(64, 64),
    se_pool='mean',
    se_attention=False,
    se_init_spread=True,                   # hot-start ON
    se_init_spread_range=(1.0, 9.0),       # moderate range (healthiest in the sweep)
    mlp_n_epochs=1000,
    mlp_learning_rate=0.0001,
    mlp_patience=20,
    mlp_device='cpu',
    batch_size=128,
    random_seeds=[42],
    test_folds=[0],
    lead_times=[12],
    save_results=True,
    experiment_phase='phase5b_hotstart_weighted',
)

# intensity-weighting strengths (step weighting; 1 = unweighted reference)
for strength in [1, 3, 10]:
    c = copy.deepcopy(base_config)
    loss = MixtureNLLLossConfig(n_components=3)
    loss.intensity_type = 'step'
    loss.intensity_strength = strength
    loss.storm_threshold = 4.5
    c.loss_config = loss                                # explicit — attaches K=3 + weighting
    c.run_name = f"p5b_hotstart1_9_wstep{strength}"
    print(f"\n=== K=3 mean-pool | hot-start [1,9] | step-weight {strength} | 12h fold0 seed42 ===")
    print(f"    confirm: n_components={c.loss_config.n_components}, "
          f"init_spread={c.se_init_spread}, range={c.se_init_spread_range}, "
          f"intensity={c.loss_config.intensity_type}×{c.loss_config.intensity_strength}")
    run_training_pipeline(c)

print("\nHot-start + weighting sweep complete.")

In [ ]:
## Hot start WITH storm weighted NLL Loss

#########################
# PHASE 5b — HOT-START + INTENSITY-WEIGHTING (K=3 mean-pool)
# Hypothesis: hot-start PLACES a component in the storm range; storm-weighted
# NLL then REWARDS keeping weight on it -> a genuine storm mode appears, instead
# of the weighting merely shifting the bulk (as it did with no storm component
# to absorb the emphasis).
# K=3, v-only, 12h, fold 0, seed 42. Hot-start [1,9] fixed; sweep weight strength.
#########################
import copy
from storm_regression.training_pipeline import (
    TrainingConfig, MixtureNLLLossConfig, run_training_pipeline
)

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    n_ensembles=200,
    filter_ensemble=False,
    ensemble_selection_method='snap',
    balance=False,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    model_type='set_encoder',
    se_phi_hidden=(64, 64),
    se_rho_hidden=(64, 64),
    se_pool='mean',
    se_attention=False,
    se_init_spread=True,                   # hot-start ON
    se_init_spread_range=(1.0, 9.0),       # moderate range (healthiest in the sweep)
    mlp_n_epochs=1000,
    mlp_learning_rate=0.0001,
    mlp_patience=20,
    mlp_device='cpu',
    batch_size=128,
    random_seeds=[42],
    test_folds=[0],
    lead_times=[12],
    save_results=True,
    experiment_phase='phase5b_hotstart_empirically_weighted_k3',
)

# intensity-weighting strengths (step weighting; 1 = unweighted reference)
for strength in [2, 3.28, 4]:
    c = copy.deepcopy(base_config)
    loss = MixtureNLLLossConfig(n_components=3)
    loss.intensity_type = 'step'
    loss.intensity_strength = strength
    loss.storm_threshold = 4.5
    c.loss_config = loss                                # explicit — attaches K=3 + weighting
    c.run_name = f"p5b_hotstart1_9_wstep{strength}"
    print(f"\n=== K=3 mean-pool | hot-start [1,9] | step-weight {strength} | 12h fold0 seed42 ===")
    print(f"    confirm: n_components={c.loss_config.n_components}, "
          f"init_spread={c.se_init_spread}, range={c.se_init_spread_range}, "
          f"intensity={c.loss_config.intensity_type}×{c.loss_config.intensity_strength}")
    run_training_pipeline(c)

print("\nEmprical Hot-start + weighting sweep complete.")

In [ ]:
## Hot start WITH storm weighted NLL Loss

#########################
# PHASE 5b — HOT-START + INTENSITY-WEIGHTING (K=3 mean-pool)
# Hypothesis: hot-start PLACES a component in the storm range; storm-weighted
# NLL then REWARDS keeping weight on it -> a genuine storm mode appears, instead
# of the weighting merely shifting the bulk (as it did with no storm component
# to absorb the emphasis).
# K=3, v-only, 12h, fold 0, seed 42. Hot-start [1,9] fixed; sweep weight strength.
#########################
import copy
from storm_regression.training_pipeline import (
    TrainingConfig, MixtureNLLLossConfig, run_training_pipeline
)

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    n_ensembles=200,
    filter_ensemble=False,
    ensemble_selection_method='snap',
    balance=False,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    model_type='set_encoder',
    se_phi_hidden=(64, 64),
    se_rho_hidden=(64, 64),
    se_pool='mean',
    se_attention=False,
    se_init_spread=True,                   # hot-start ON
    se_init_spread_range=(1.0, 9.0),       # moderate range (healthiest in the sweep)
    mlp_n_epochs=1000,
    mlp_learning_rate=0.0001,
    mlp_patience=20,
    mlp_device='cpu',
    batch_size=128,
    random_seeds=[42],
    test_folds=[0],
    lead_times=[12],
    save_results=True,
    experiment_phase='phase5b_hotstart_empirically_weighted_k2',
)

# intensity-weighting strengths (step weighting; 1 = unweighted reference)
for strength in [2, 3.28, 4]:
    c = copy.deepcopy(base_config)
    loss = MixtureNLLLossConfig(n_components=2)
    loss.intensity_type = 'step'
    loss.intensity_strength = strength
    loss.storm_threshold = 4.5
    c.loss_config = loss                                # explicit — attaches K=3 + weighting
    c.run_name = f"p5b_hotstart1_9_wstep{strength}"
    print(f"\n=== K=3 mean-pool | hot-start [1,9] | step-weight {strength} | 12h fold0 seed42 ===")
    print(f"    confirm: n_components={c.loss_config.n_components}, "
          f"init_spread={c.se_init_spread}, range={c.se_init_spread_range}, "
          f"intensity={c.loss_config.intensity_type}×{c.loss_config.intensity_strength}")
    run_training_pipeline(c)

print("\nEmprical Hot-start + weighting sweep complete.")

## Attention layer

In [ ]:
#########################
# PHASE 5b — ATTENTION POOLING (K=3) — the learned per-member weighting (§5.3)
# Three runs: (a) no hot-start, (b) hot-start [1,9], (c) hot-start + wstep2.
# Everything else identical to the mean-pool runs so the ONLY change vs mean-pool
# is se_attention=True. 12h, fold 0, seed 42.
#########################
import copy
from storm_regression.training_pipeline import (
    TrainingConfig, MixtureNLLLossConfig, run_training_pipeline
)

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    n_ensembles=200,
    filter_ensemble=False,
    ensemble_selection_method='snap',
    balance=False,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    model_type='set_encoder',
    se_phi_hidden=(64, 64),
    se_rho_hidden=(64, 64),
    se_pool='mean',                 # ignored when attention=True, but set for clarity
    se_attention=True,              # <-- THE CHANGE: learned per-member weighting
    se_init_spread_range=(1.0, 9.0),
    mlp_n_epochs=1000,
    mlp_learning_rate=0.0001,
    mlp_patience=40,                # slightly more patience — attention trains slower
    mlp_device='cpu',
    batch_size=128,
    random_seeds=[42],
    test_folds=[0],
    lead_times=[12],
    save_results=True,
    experiment_phase='phase5b_attention',
)

# (label, hot_start, weight_strength)
runs = [
    ('nohot',        False, 1),      # attention, no hot-start, unweighted
    ('hot1_9',       True,  1),      # attention + hot-start, unweighted
    ('hot1_9_wstep2', True, 2),      # attention + hot-start + weighted loss
]

for label, hot, w in runs:
    c = copy.deepcopy(base_config)
    c.se_init_spread = hot
    loss = MixtureNLLLossConfig(n_components=3)
    loss.intensity_type = 'step' if w != 1 else 'none'
    loss.intensity_strength = w
    loss.storm_threshold = 4.5
    c.loss_config = loss
    c.run_name = f"p5b_attn_k3_{label}"
    print(f"\n=== ATTENTION K=3 | {label} | hot_start={hot} weight={w} | 12h fold0 seed42 ===")
    print(f"    confirm: attention={c.se_attention}, n_components={c.loss_config.n_components}, "
          f"init_spread={c.se_init_spread}, intensity={c.loss_config.intensity_type}×{c.loss_config.intensity_strength}")
    run_training_pipeline(c)

print("\nAttention pooling runs complete.")

In [ ]:
#########################
# PHASE 5b — ATTENTION POOLING (K=3) — the learned per-member weighting (§5.3)
# Three runs: (a) no hot-start, (b) hot-start [1,9], (c) hot-start + wstep2.
# Everything else identical to the mean-pool runs so the ONLY change vs mean-pool
# is se_attention=True. 12h, fold 0, seed 42.
#########################
import copy
from storm_regression.training_pipeline import (
    TrainingConfig, MixtureNLLLossConfig, run_training_pipeline
)

base_config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    n_ensembles=200,
    filter_ensemble=False,
    ensemble_selection_method='snap',
    balance=False,
    remove_cmes=True,
    omni_subset=['Bz_GSM', 'Dst', 'n_sw'],
    model_type='set_encoder',
    se_phi_hidden=(64, 64),
    se_rho_hidden=(64, 64),
    se_pool='mean',                 # ignored when attention=True, but set for clarity
    se_attention=True,              # <-- THE CHANGE: learned per-member weighting
    se_init_spread_range=(1.0, 9.0),
    mlp_n_epochs=1000,
    mlp_learning_rate=0.0001,
    mlp_patience=40,                # slightly more patience — attention trains slower
    mlp_device='cpu',
    batch_size=128,
    random_seeds=[42],
    test_folds=[0],
    lead_times=[12],
    save_results=True,
    experiment_phase='phase5b_attention',
)

# (label, hot_start, weight_strength)
runs = [
    ('nohot',        False, 1),      # attention, no hot-start, unweighted
    ('hot1_9',       True,  1),      # attention + hot-start, unweighted
    ('hot1_9_wstep2', True, 2),      # attention + hot-start + weighted loss
]

for label, hot, w in runs:
    c = copy.deepcopy(base_config)
    c.se_init_spread = hot
    loss = MixtureNLLLossConfig(n_components=2)
    loss.intensity_type = 'step' if w != 1 else 'none'
    loss.intensity_strength = w
    loss.storm_threshold = 4.5
    c.loss_config = loss
    c.run_name = f"p5b_attn_k2_{label}"
    print(f"\n=== ATTENTION K=2 | {label} | hot_start={hot} weight={w} | 12h fold0 seed42 ===")
    print(f"    confirm: attention={c.se_attention}, n_components={c.loss_config.n_components}, "
          f"init_spread={c.se_init_spread}, intensity={c.loss_config.intensity_type}×{c.loss_config.intensity_strength}")
    run_training_pipeline(c)

print("\nAttention pooling runs complete.")

### Checking attention weights for a single run

Can't run this through training pipeline since we need to the model output - easier just to run locally

In [ ]:
import torch, numpy as np
from torch.utils.data import Subset, DataLoader
from storm_utils.data_structure import ForecastingDataset
from storm_regression.training_pipeline import (
    train_set_encoder, prepare_set_encoder_features, TrainingConfig, MixtureNLLLossConfig
)

# rebuild the config for the run you want to inspect (attention K=2 hot-start, say)
loss = MixtureNLLLossConfig(n_components=2); loss.intensity_type='none'
config = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    n_ensembles=200, filter_ensemble=False, ensemble_selection_method='snap',
    balance=False, remove_cmes=True, omni_subset=['Bz_GSM','Dst','n_sw'],
    model_type='set_encoder', se_phi_hidden=(64,64), se_rho_hidden=(64,64),
    se_pool='mean', se_attention=True, se_init_spread=True, se_init_spread_range=(1.0,9.0),
    mlp_n_epochs=1000, mlp_learning_rate=1e-4, mlp_patience=30, mlp_device='cpu',
    batch_size=128, loss_config=loss,
)

# rebuild dataset + split exactly as the pipeline does
dataset = ForecastingDataset(
    parquet_path=str(config.huxt_data_path),
    discontinuity_path=str(config.discontinuity_path),
    seed=42, Nens=200, lead_time_hours=12,
    forecast_duration_hours=config.forecast_duration_hours, stride_hours=config.stride_hours)
dataset.remove_cmes(inplace=True)
tr_idx, val_idx, te_idx = dataset.rotation_aligned_train_val_test_split(test_fold=0, n_groups=5, buffer_margin_hours=0.0)
dataset.set_omni_columns(config.omni_subset)

tr = DataLoader(Subset(dataset, tr_idx), batch_size=128, shuffle=True)
val = DataLoader(Subset(dataset, val_idx), batch_size=128, shuffle=False)

# train — model stays in scope
model, scalers, history = train_set_encoder(
    tr, val, 200, loss_config=loss, config=config,
    device='cpu', patience=30, n_epochs=1000, learning_rate=1e-4, batch_size=128)

# now inspect attention on a test batch
ctx_scaler, v_scaler = scalers
test_dl = DataLoader(Subset(dataset, te_idx), batch_size=256, shuffle=False)
batch = next(iter(test_dl))
v_m, ctx, _, _ = prepare_set_encoder_features(batch, 200, scaler=ctx_scaler, v_scaler=v_scaler)
v_t = torch.as_tensor(v_m, dtype=torch.float32)
c_t = torch.as_tensor(ctx, dtype=torch.float32)

model.eval()
with torch.no_grad():
    B, Nens, T = v_t.shape
    e = model.phi(v_t.reshape(B*Nens, T)).reshape(B, Nens, model.h)
    logits = model.attn(e).squeeze(-1)            # (B, Nens) raw scores
    weights = torch.softmax(logits, dim=1).cpu().numpy()   # (B, Nens)

uniform = 1.0/Nens
entropy = -(weights*np.log(weights+1e-12)).sum(1)
eff_n = np.exp(entropy)
print(f"uniform weight = {uniform:.4f}")
print(f"logit range: {logits.min():.3f} to {logits.max():.3f}  (near-0 = uniform)")
print(f"weight range: {weights.min():.4f} to {weights.max():.4f}")
print(f"effective # members attended (mean): {eff_n.mean():.1f} / {Nens}")

# 2. Effective number of members attended to (entropy-based).
#    If ~Nens, it's basically uniform (=mean pooling). If ~1, it's spiking.
entropy = -(weights * np.log(weights + 1e-12)).sum(1)          # (B,)
eff_n = np.exp(entropy)                              # effective # members per forecast
print(f"effective # members attended (mean): {eff_n.mean():.1f} out of {Nens}")
print(f"  (near {Nens} = uniform/mean-pooling; near 1 = attends to one member)")

# 3. Plot the distribution of weights for a few forecasts
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for i in range(min(5, B)):
    ax[0].plot(np.sort(weights[i])[::-1], alpha=0.6)      # sorted weights per forecast
ax[0].axhline(uniform, color='k', ls='--', label='uniform')
ax[0].set_title('Sorted attention weights (5 forecasts)')
ax[0].set_xlabel('member rank'); ax[0].set_ylabel('weight'); ax[0].legend()
ax[1].hist(eff_n, bins=40)
ax[1].axvline(Nens, color='k', ls='--', label=f'uniform ({Nens})')
ax[1].set_title('Effective # members attended per forecast')
ax[1].set_xlabel('effective N'); ax[1].legend()
plt.tight_layout(); plt.savefig('attention_weights.png', dpi=120); plt.close()

In [ ]:
# 3. Plot the distribution of weights for a few forecasts, coloured by storm/quiet
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
l = min(100, B)
is_storm = y[:B] > 4.5              # match the forecasts being plotted

for i in range(l):
    c = 'crimson' if is_storm[i] else 'steelblue'
    ax[0].plot(np.sort(weights[i])[::-1], alpha=0.3, color=c)   # sorted weights per forecast
ax[0].axhline(uniform, color='k', ls='--', label='uniform')
# legend proxies (so storm/quiet show once, not per-curve)
from matplotlib.lines import Line2D
ax[0].legend(handles=[
    Line2D([0], [0], color='crimson', label='storm'),
    Line2D([0], [0], color='steelblue', label='quiet'),
    Line2D([0], [0], color='k', ls='--', label='uniform'),
])
ax[0].set_title('Sorted attention weights (storm vs quiet)')
ax[0].set_xlabel('member rank'); ax[0].set_ylabel('weight')

ax[1].hist(eff_n, bins=25)
ax[1].axvline(Nens, color='k', ls='--', label=f'uniform ({Nens})')
ax[1].set_title('Effective # members attended per forecast')
ax[1].set_xlabel('effective N'); ax[1].legend()
plt.tight_layout(); plt.savefig('attention_weights.png', dpi=120); plt.show()

## Self attention model

In [ ]:
#########################
# PHASE 5b — SELF-ATTENTION (inter-member) via run_training_pipeline
# Proper run: same dataset/split/scaler/eval path as all other runs, so it's
# comparable to mean-pool and isolated-attention at the matched config.
# K=3, hot-start [1,9], wstep2, 12h, fold 0, seed 42.
#########################
import copy
from storm_regression.training_pipeline import (
    TrainingConfig, MixtureNLLLossConfig, run_training_pipeline
)

base = TrainingConfig(
    huxt_run_id=3,
    huxt_data_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'full_df.parquet',
    discontinuity_path=paths['huxt_data_shared'] / 'HUXt3_modified' / 'discontinuities.npy',
    n_ensembles=50, filter_ensemble=False, ensemble_selection_method='snap',
    balance=False, remove_cmes=True, omni_subset=['Bz_GSM','Dst','n_sw'],
    model_type='set_encoder', se_phi_hidden=(64,64), se_rho_hidden=(64,64),
    se_self_attention=True, se_sa_heads=4, se_sa_layers=1,
    se_init_spread=True, se_init_spread_range=(1.0,9.0),
    mlp_n_epochs=1000, mlp_learning_rate=1e-4, mlp_patience=30, mlp_device='cpu',
    batch_size=128, random_seeds=[42], test_folds=[0], lead_times=[12],
    save_results=True, experiment_phase='phase5b_selfattn',
)

# two configs: self-attn + mean pool (clean skill), self-attn + attention pool (interpretable weights)
variants = [
    ('meanpool', False),
    ('attnpool', True),
]
for label, attn_pool in variants:
    c = copy.deepcopy(base)
    c.se_pool = 'mean'
    c.se_attention = attn_pool
    loss = MixtureNLLLossConfig(n_components=3)
    loss.intensity_type = 'step'; loss.intensity_strength = 2; loss.storm_threshold = 4.5
    c.loss_config = loss
    c.run_name = f'p5b_selfattn_{label}_k3_hot1_9_wstep2'
    print(f"\n=== SELF-ATTN + {label} | K=3 hot[1,9] wstep2 | 12h fold0 seed42 ===")
    print(f"    confirm: self_attn={c.se_self_attention}, attn_pool={c.se_attention}, "
          f"K={c.loss_config.n_components}, init_spread={c.se_init_spread}, "
          f"intensity={c.loss_config.intensity_type}×{c.loss_config.intensity_strength}")
    run_training_pipeline(c)

print("\nSelf-attention runs complete.")